In [2]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch

In [10]:
DELTA = 4.0 
MOT_W, MOT_H = 35.0, 42.0 
RACK_W = 8.0 
RACK_H_FULL = 70.0 
RACK_H_CASING = 69.0  # Рабочая длина рейки в корпусе

# Радиус описанной вокруг мотора окружности (для проверки зазора с D40)
R_MOT = np.hypot(MOT_W / 2.0, MOT_H / 2.0)

PENALTY_WEIGHT = 3000

In [11]:
def layout(theta): #theta - матрица весов, в нашем случае углы расстояний между колесами
    alpha, beta, gamma = theta[0], theta[1], theta[2]

    x1 = torch.tensor(0.0)
    y1 = torch.tensor(0.0) #зафиксирвоанное положение первого колеса

    x2 = x1 + 15.0 * torch.cos(alpha)
    y2 = y1 + 15.0 * torch.sin(alpha)

    x3 = x2 + 20.0 * torch.cos(beta)
    y3 = y2 + 20.0 * torch.sin(beta)

    x4 = x3 + 30.0 * torch.cos(gamma) #сказали больше комменатриев делать но тут врод ясно ды
    y4 = y3 + 30.0 * torch.sin(gamma) 

    rack_x1 = x4 + 20.0
    rack_x2 = rack_x1 + RACK_W

    rack_casing = (rack_x1, rack_x2, y4 - RACK_H_CASING / 2.0, y4 + RACK_H_CASING / 2.0)
    rack_full = (rack_x1, rack_x2, y4 - RACK_H_FULL / 2.0, y4 + RACK_H_FULL / 2.0)

    axes = ((x1, y1), (x2, y2), (x3, y3), (x4, y4))
    return axes, rack_casing, rack_full
    

In [12]:
def get_bounding_box(axes, rack_casing):
    (x1, y1), (x2, y2), (x3, y3), (x4, y4) = axes
    rx1, rx2, ry1_cas, ry2_cas = rack_casing
    #по факту дальше мы закинем все точки в массив, а в нем найдем крайние точки и там площадь бах бах 
    xs_min = torch.stack(
        [-torch.tensor(MOT_W / 2.0), x2 - 10.0, x3 - 15.0, x4 - 25.0, rx1]
    )

    xs_max = torch.stack(
        [torch.tensor(MOT_W / 2.0), x2 + 10.0, x3 + 15.0, x4 + 25.0, rx2]
    )

    ys_min = torch.stack(
        [-torch.tensor(MOT_H / 2.0), y2 - 10.0, y3 - 15.0, y4 - 25.0, ry1_cas]
    )

    ys_max = torch.stack(
        [torch.tensor(MOT_H / 2.0), y2 + 10.0, y3 + 15.0, y4 + 25.0, ry2_cas]
    )
    width = torch.max(xs_max) - torch.min(xs_min)
    height = torch.max(ys_max) - torch.min(ys_min) 

    return width, height

In [13]:
def check_collisions(axes, rack_full):
    (x1, y1), (x2, y2), (x3, y3), (x4, y4) = axes
    rx1, rx2, ry1_full, ry2_full = rack_full

    d13 = torch.hypot(x3 - x1, y3 - y1)
    loss_13 = torch.relu(24.0 - d13) ** 2 #вы поняли да если коллизия то релушечка родная даст ошибку хорошую

    d24 = torch.hypot(x4 - x2, y4 - y2)
    loss_24 = torch.relu(39.0 - d24) ** 2

    dx = torch.maximum(
        -torch.tensor(MOT_W / 2.0) - rx2, rx1 - torch.tensor(MOT_W / 2.0)
    )
    dy = torch.maximum(
        -torch.tensor(MOT_H / 2.0) - ry2_full,
        ry1_full - torch.tensor(MOT_H / 2.0),
    )

    # короче параллельные прямоугольники не персекаются если их разделяет прямая по какой либо из осей
    gap = torch.maximum(dx, dy)
    loss_rack_mot = torch.relu(DELTA - gap) ** 2

    d14 = torch.hypot(x4 - x1, y4 - y1)
    loss_mot_c4 = torch.relu((20.0 + R_MOT + DELTA) - d14) ** 2

    # Суммарный штраф
    total_penalty = loss_13 + loss_24 + loss_rack_mot + loss_mot_c4
    return total_penalty

In [14]:
def compute_loss(theta):
    axes, rack_casing, rack_full = layout(theta)
    width, height = get_bounding_box(axes, rack_casing)
    raw_penalty = check_collisions(axes, rack_full)
    
    area = width * height

    total_loss = area + PENALTY_WEIGHT * raw_penalty

    return total_loss, area, raw_penalty, (width, height)

In [15]:
# theta = torch.nn.Parameter(torch.rand(3) * 2 * np.pi) 
## так как у нас релу то He не пойдет мы вот да вот эту тему 
theta_tensor = torch.empty(1, 3)
torch.nn.init.kaiming_uniform_(theta_tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')
theta = torch.nn.Parameter(theta_tensor.squeeze(0))

optimizer = torch.optim.Adam([theta], lr=0.03)

print("Запуск оптимизации компоновки...")

for epoch in range(801):
    optimizer.zero_grad()

    loss, area, penalty, (w, h) = compute_loss(theta)

    loss.backward()

    optimizer.step()
    # scheduler.step()

    if epoch % 200 == 0:
        print(
            f"Эпоха {epoch:3d} | Loss: {loss.item():.1f} | Площадь: {area.item():.1f} мм² | Штраф: {penalty.item():.4f}"
        )

with torch.no_grad():
    _, final_area, final_pen, (final_w, final_h) = compute_loss(theta)
    final_deg = (theta.numpy() * 180 / np.pi) % 360

print("\n=== ГОТОВО ===")
print(
    f"Оптимальные углы: α={final_deg[0]:.1f}°, β={final_deg[1]:.1f}°, γ={final_deg[2]:.1f}°"
)
print(f"Габариты: {final_w.item():.2f} × {final_h.item():.2f} мм")
print(f"Площадь корпуса: {final_area.item():.1f} мм²")
print(f"Коллизии устранены: {final_pen.item() < 1e-3}")

Запуск оптимизации компоновки...
Эпоха   0 | Loss: 8420.4 | Площадь: 8420.4 мм² | Штраф: 0.0000
Эпоха 200 | Loss: 6557.4 | Площадь: 6557.1 мм² | Штраф: 0.0001
Эпоха 400 | Loss: 6557.6 | Площадь: 6557.4 мм² | Штраф: 0.0001
Эпоха 600 | Loss: 6558.7 | Площадь: 6558.6 мм² | Штраф: 0.0000
Эпоха 800 | Loss: 6572.6 | Площадь: 6572.6 мм² | Штраф: 0.0000

=== ГОТОВО ===
Оптимальные углы: α=54.8°, β=44.6°, γ=334.7°
Габариты: 95.53 × 69.00 мм
Площадь корпуса: 6591.2 мм²
Коллизии устранены: True
